# **Import Libraries & Load Dataset**

In [15]:

import pandas as pd
import numpy as np

from google.colab import files

uploaded = files.upload()

file_name = "money_laundring_cleaned_and_feature_engineered.csv"
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())


Saving money_laundring_cleaned_and_feature_engineered.csv to money_laundring_cleaned_and_feature_engineered (2).csv
Dataset loaded successfully!
Dataset shape: (97066, 14)


,Day,hour,Sender_frequancy,Reciever_frequancy,Amount,Payment_currency,Received_currency,is_payment_same,Sender_bank_location,Receiver_bank_location,is_bank_same_location,Payment_type,Is_laundering,Laundering_type
0,Friday,10,5,1,92.61,Euros,UK pounds,False,UK,UK,True,Credit card,0,Normal_Small_Fan_Out
1,Tuesday,19,110,25,5157.16,UK pounds,UK pounds,True,UK,Japan,False,Cross-border,1,Behavioural_Change_2
2,Saturday,8,14,3,7426.99,UK pounds,UK pounds,True,UAE,UK,False,Cheque,0,Normal_Fan_Out
3,Tuesday,15,2,1,3391.18,UK pounds,UK pounds,True,US,Nigeria,False,ACH,0,Normal_Small_Fan_Out
4,Saturday,4,2,20,6951.29,US dollars,US dollars,True,UK,Germany,False,Debit card,0,Normal_Fan_In


## **Data Quality Check**

In [16]:

print("Dataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["Is_laundering"].value_counts())

print("\nTarget percentages:")
print(
    (df["Is_laundering"].value_counts(normalize=True) * 100).round(2)
)


Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97066 entries, 0 to 97065
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Day                     97066 non-null  object 
 1   hour                    97066 non-null  int64  
 2   Sender_frequancy        97066 non-null  int64  
 3   Reciever_frequancy      97066 non-null  int64  
 4   Amount                  97066 non-null  float64
 5   Payment_currency        97066 non-null  object 
 6   Received_currency       97066 non-null  object 
 7   is_payment_same         97066 non-null  bool   
 8   Sender_bank_location    97066 non-null  object 
 9   Receiver_bank_location  97066 non-null  object 
 10  is_bank_same_location   97066 non-null  bool   
 11  Payment_type            97066 non-null  object 
 12  Is_laundering           97066 non-null  int64  
 13  Laundering_type         97066 non-null  object 
dtypes: bool(2), float

## **Clean Data & Prepare Features**

In [17]:

# Remove duplicate rows
df = df.drop_duplicates().copy()

# Remove rows with missing target
df = df.dropna(subset=["Is_laundering"]).copy()

# Validate target values
df["Is_laundering"] = pd.to_numeric(
    df["Is_laundering"], errors="coerce"
)
df = df.dropna(subset=["Is_laundering"]).copy()
df = df[df["Is_laundering"].isin([0, 1])].copy()
df["Is_laundering"] = df["Is_laundering"].astype(int)

# Convert Boolean-like columns
boolean_columns = [
    "is_payment_same",
    "is_bank_same_location"
]

for col in boolean_columns:
    if col in df.columns:
        df[col] = df[col].replace({
            True: 1,
            False: 0,
            "True": 1,
            "False": 0,
            "true": 1,
            "false": 0
        })

        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Data cleaning completed.")
print("Final shape:", df.shape)
print("Target missing values:", df["Is_laundering"].isna().sum())


Data cleaning completed.
Final shape: (97065, 14)
Target missing values: 0


/tmp/ipykernel_9792/2607618036.py:23: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace({


## **Explore Target Leakage**

In [18]:

# Check the relationship between laundering type and target
if "Laundering_type" in df.columns:
    laundering_type_analysis = pd.crosstab(
        df["Laundering_type"],
        df["Is_laundering"],
        normalize="index"
    ).round(3)

    display(laundering_type_analysis)

# Check whether frequency features exist
frequency_columns = [
    "Sender_frequancy",
    "Reciever_frequancy"
]

print(
    "Available frequency features:",
    [col for col in frequency_columns if col in df.columns]
)


Is_laundering,0,1
Laundering_type,,
Behavioural_Change_1,0.000,1.000
Behavioural_Change_2,0.000,1.000
Bipartite,0.000,1.000
Cash_Withdrawal,0.000,1.000
Cycle,0.000,1.000
Deposit-Send,0.000,1.000
Fan_In,0.000,1.000
Fan_Out,0.000,1.000
Gather-Scatter,0.000,1.000


Available frequency features: ['Sender_frequancy', 'Reciever_frequancy']


##**Feature Engineering**


In [38]:

# Convert hour into cyclical features
if "hour" in df.columns:
    df["hour"] = pd.to_numeric(df["hour"], errors="coerce")

    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Ensure Amount is numeric
if "Amount" in df.columns:
    df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")
    df["Amount"] = df["Amount"].clip(lower=0)

print("Feature engineering completed.")
display(df.head())


Feature engineering completed.


,Day,hour,Sender_frequancy,Reciever_frequancy,Amount,Payment_currency,Received_currency,is_payment_same,Sender_bank_location,Receiver_bank_location,is_bank_same_location,Payment_type,Is_laundering,Laundering_type,hour_sin,hour_cos
0,Friday,10,5,1,92.61,Euros,UK pounds,0,UK,UK,1,Credit card,0,Normal_Small_Fan_Out,0.500000,-0.866025
1,Tuesday,19,110,25,5157.16,UK pounds,UK pounds,1,UK,Japan,0,Cross-border,1,Behavioural_Change_2,-0.965926,0.258819
2,Saturday,8,14,3,7426.99,UK pounds,UK pounds,1,UAE,UK,0,Cheque,0,Normal_Fan_Out,0.866025,-0.500000
3,Tuesday,15,2,1,3391.18,UK pounds,UK pounds,1,US,Nigeria,0,ACH,0,Normal_Small_Fan_Out,-0.707107,-0.707107
4,Saturday,4,2,20,6951.29,US dollars,US dollars,1,UK,Germany,0,Debit card,0,Normal_Fan_In,0.866025,0.500000


## **Define Features & Target**

In [20]:

target_column = "Is_laundering"

columns_to_exclude = [
    "Is_laundering",
    "Laundering_type",
    "hour",
    "Sender_frequancy",
    "Reciever_frequancy"
]

X = df.drop(
    columns=columns_to_exclude,
    errors="ignore"
).copy()

y = df[target_column].copy()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)
print("Number of features before encoding:", X.shape[1])
print("Target in X:", target_column in X.columns)


Categorical features: ['Day', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type']
Numerical features: ['Amount', 'is_payment_same', 'is_bank_same_location', 'hour_sin', 'hour_cos']
Number of features before encoding: 11
Target in X: False


## **Split Data into Train, Validation & Test**

In [21]:

from sklearn.model_selection import train_test_split

# First split: 80% development, 20% final test
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: 75% of development for training, 25% for validation
# Overall: 60% train, 20% validation, 20% test
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nValidation target distribution:")
print(y_val.value_counts(normalize=True).round(3))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(3))


Training set: (58239, 11)
Validation set: (19413, 11)
Test set: (19413, 11)

Training target distribution:
Is_laundering
0    0.768
1    0.232
Name: proportion, dtype: float64

Validation target distribution:
Is_laundering
0    0.768
1    0.232
Name: proportion, dtype: float64

Test target distribution:
Is_laundering
0    0.768
1    0.232
Name: proportion, dtype: float64


## **Build Preprocessing Pipeline**

In [22]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created.")


Preprocessing pipeline created.


# **Train Random Forest Baseline**

In [23]:

from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")


Random Forest training completed.


## **Evaluate Random Forest on Validation Data**

In [24]:

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

rf_val_pred = rf_model.predict(X_val)
rf_val_prob = rf_model.predict_proba(X_val)[:, 1]

print("Validation Confusion Matrix:")
print(confusion_matrix(y_val, rf_val_pred))

print("\nValidation Classification Report:")
print(classification_report(
    y_val,
    rf_val_pred,
    target_names=["Legitimate", "Laundering"],
    digits=4,
    zero_division=0
))

print(
    "Validation PR-AUC (Average Precision):",
    round(average_precision_score(y_val, rf_val_prob), 4)
)

print(
    "Validation ROC-AUC:",
    round(roc_auc_score(y_val, rf_val_prob), 4)
)


Validation Confusion Matrix:
[[14151   766]
 [ 2509  1987]]

Validation Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.8494    0.9486    0.8963     14917
  Laundering     0.7218    0.4419    0.5482      4496

    accuracy                         0.8313     19413
   macro avg     0.7856    0.6953    0.7222     19413
weighted avg     0.8198    0.8313    0.8157     19413

Validation PR-AUC (Average Precision): 0.6817
Validation ROC-AUC: 0.882


## **Compare Decision Thresholds on Validation Data**

In [25]:

from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50]:
    val_pred_threshold = (
        rf_val_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, val_pred_threshold, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_pred_threshold, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, val_pred_threshold, zero_division=0
        ),
        "False Positives": int(
            confusion_matrix(
                y_val, val_pred_threshold, labels=[0, 1]
            )[0, 1]
        ),
        "False Negatives": int(
            confusion_matrix(
                y_val, val_pred_threshold, labels=[0, 1]
            )[1, 0]
        )
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df.round(4))


,Threshold,Precision,Recall,F1-score,False Positives,False Negatives
0,0.30,0.5782,0.7389,0.6488,2423,1174
1,0.35,0.6202,0.6599,0.6394,1817,1529
2,0.40,0.6569,0.5856,0.6192,1375,1863
3,0.45,0.6923,0.5149,0.5906,1029,2181
4,0.50,0.7193,0.4486,0.5526,787,2479


## **Select a Threshold**

In [26]:

minimum_recall = 0.75

eligible_thresholds = threshold_df[
    threshold_df["Recall"] >= minimum_recall
]

if not eligible_thresholds.empty:
    selected_row = eligible_thresholds.sort_values(
        by=["Precision", "F1-score"],
        ascending=False
    ).iloc[0]

    best_threshold = float(selected_row["Threshold"])
else:
    # Fallback: threshold with the highest validation F1-score
    selected_row = threshold_df.loc[
        threshold_df["F1-score"].idxmax()
    ]

    best_threshold = float(selected_row["Threshold"])

print("Selected threshold:", best_threshold)
print("\nSelected validation metrics:")
print(selected_row)


Selected threshold: 0.3

Selected validation metrics:
Threshold             0.300000
Precision             0.578242
Recall                0.738879
F1-score              0.648765
False Positives    2423.000000
False Negatives    1174.000000
Name: 0, dtype: float64


## **Train XGBoost**

In [27]:

%pip -q install xgboost


In [28]:

from xgboost import XGBClassifier

xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train)

print("XGBoost training completed.")


XGBoost training completed.


## **Compare Random Forest & XGBoost on Validation Data**

In [30]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

models = {
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

comparison_results = []

for model_name, trained_model in models.items():
    val_prob = trained_model.predict_proba(X_val)[:, 1]

    # Compare at the same threshold first
    val_pred = (val_prob >= 0.30).astype(int)

    comparison_results.append({
        "Model": model_name,
        "Precision": precision_score(
            y_val, val_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_pred, zero_division=0
        ),
        "F1-score": f1_score(
            y_val, val_pred, zero_division=0
        ),
        "PR-AUC": average_precision_score(y_val, val_prob),
        "ROC-AUC": roc_auc_score(y_val, val_prob)
    })

comparison_df = pd.DataFrame(comparison_results)
display(comparison_df.round(4))


,Model,Precision,Recall,F1-score,PR-AUC,ROC-AUC
0,Random Forest,0.5782,0.7389,0.6488,0.6817,0.8820
1,XGBoost,0.6292,0.6964,0.6611,0.7228,0.8957


## **Final Test Evaluation**

In [31]:

# Choose the model based on validation results.
# Change this to "XGBoost" if it performs better for your objective.
selected_model_name = "Random Forest"
final_model = models[selected_model_name]

# Use the threshold selected from validation for this model.
# If you selected a different threshold for XGBoost, update it accordingly.
final_threshold = best_threshold

# Final predictions on the untouched test set
test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= final_threshold).astype(int)

print("Selected model:", selected_model_name)
print("Selected threshold:", final_threshold)

print("\nFinal Test Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nFinal Test Classification Report:")
print(classification_report(
    y_test,
    test_pred,
    target_names=["Legitimate", "Laundering"],
    digits=4,
    zero_division=0
))

print(
    "Final Test PR-AUC:",
    round(average_precision_score(y_test, test_prob), 4)
)

print(
    "Final Test ROC-AUC:",
    round(roc_auc_score(y_test, test_prob), 4)
)


Selected model: Random Forest
Selected threshold: 0.3

Final Test Confusion Matrix:
[[12458  2459]
 [ 1161  3335]]

Final Test Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9148    0.8352    0.8731     14917
  Laundering     0.5756    0.7418    0.6482      4496

    accuracy                         0.8135     19413
   macro avg     0.7452    0.7885    0.7607     19413
weighted avg     0.8362    0.8135    0.8210     19413

Final Test PR-AUC: 0.6889
Final Test ROC-AUC: 0.882


## **Save the Trained Model**

In [32]:

import joblib

joblib.dump(final_model, "aml_detection_model.joblib")

print("Model saved successfully: aml_detection_model.joblib")

files.download("aml_detection_model.joblib")


Model saved successfully: aml_detection_model.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:
import joblib
from google.colab import files
import os

# Save the selected threshold
best_threshold = 0.30

joblib.dump(best_threshold, "aml_detection_threshold.joblib")

print("Threshold saved:", os.path.exists("aml_detection_threshold.joblib"))
files.download("aml_detection_threshold.joblib")

Threshold saved: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
import os

print("Random Forest:", os.path.exists("aml_detection_model.joblib"))
print("Threshold:", os.path.exists("aml_detection_threshold.joblib"))

Random Forest: True
Threshold: True


In [37]:
import joblib

# Load the saved model and threshold
loaded_model = joblib.load("aml_detection_model.joblib")
loaded_threshold = joblib.load("aml_detection_threshold.joblib")

print("Model loaded successfully:", type(loaded_model).__name__)
print("Selected threshold:", loaded_threshold)

Model loaded successfully: Pipeline
Selected threshold: 0.3
